In [ ]:
!pip -q install streamlit mcp google-genai google-api-python-client youtube-transcript-api requests markdown
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
import os
from google.colab import userdata
#os.environ["APP_PASSWORD"] = userdata.get("APP_PASSWORD")  # remove this line if not using a password

In [ ]:
%%writefile server.py
import os, requests, markdown
from mcp.server.fastmcp import FastMCP
from googleapiclient.discovery import build
from googleapiclient.http import MediaInMemoryUpload
from google.oauth2.credentials import Credentials
from youtube_transcript_api import YouTubeTranscriptApi

mcp = FastMCP("learning-agent")

@mcp.tool()
def youtube_search(query: str, max_results: int = 5) -> list:
    """Search YouTube for tutorials. Returns title, channel, url, video_id."""
    yt = build("youtube", "v3", developerKey=os.environ["YOUTUBE_API_KEY"])
    r = yt.search().list(q=query, part="snippet", type="video",
                         maxResults=max_results, relevanceLanguage="en").execute()
    return [{"title": i["snippet"]["title"], "channel": i["snippet"]["channelTitle"],
             "video_id": i["id"]["videoId"],
             "url": f"https://youtu.be/{i['id']['videoId']}"} for i in r["items"]]

@mcp.tool()
def youtube_transcript(video_id: str) -> str:
    """Get the transcript text of a YouTube video (first ~6000 chars)."""
    try:
        t = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join(x["text"] for x in t)[:6000]
    except Exception as e:
        return f"No transcript: {e}"

@mcp.tool()
def drive_create_doc(title: str, content: str) -> str:
    """Create a formatted Google Doc from markdown text. Returns its link."""
    creds = Credentials(token=os.environ["DRIVE_ACCESS_TOKEN"])
    drive = build("drive", "v3", credentials=creds)
    html = markdown.markdown(content, extensions=["extra"])
    media = MediaInMemoryUpload(html.encode(), mimetype="text/html")
    f = drive.files().create(
        body={"name": title, "mimeType": "application/vnd.google-apps.document"},
        media_body=media, fields="id,webViewLink").execute()
    return f["webViewLink"]

@mcp.tool()
def notion_create_page(title: str, content: str) -> str:
    """Create a Notion page from markdown-like text (#, ##, - [ ], - bullets). Returns its URL."""
    def block(line):
        l = line.strip()
        if l.startswith("## "):  t, k = l[3:], "heading_3"
        elif l.startswith("# "): t, k = l[2:], "heading_2"
        elif l.startswith("- [ ]") or l.startswith("- [x]"):
            return {"object": "block", "type": "to_do", "to_do": {
                "rich_text": [{"type": "text", "text": {"content": l[5:].strip()[:1900]}}],
                "checked": l.startswith("- [x]")}}
        elif l.startswith("- "): t, k = l[2:], "bulleted_list_item"
        else: t, k = l, "paragraph"
        return {"object": "block", "type": k,
                k: {"rich_text": [{"type": "text", "text": {"content": t[:1900]}}]}}
    blocks = [block(l) for l in content.split("\n") if l.strip()][:100]
    r = requests.post("https://api.notion.com/v1/pages",
        headers={"Authorization": f"Bearer {os.environ['NOTION_TOKEN']}",
                 "Notion-Version": "2022-06-28"},
        json={"parent": {"page_id": os.environ["NOTION_PARENT_PAGE_ID"]},
              "properties": {"title": {"title": [{"text": {"content": title}}]}},
              "children": blocks})
    r.raise_for_status()
    return r.json()["url"]

if __name__ == "__main__":
    mcp.run()

Writing server.py


In [ ]:
%%writefile app.py
import asyncio, os, streamlit as st
from google import genai
from google.genai import types
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MODEL = "gemini-3.7-flash"
DEST = {"Notion": "notion_create_page", "Google Drive": "drive_create_doc"}

SYSTEM = """You are a programming learning agent.
For any programming-language or tech topic:
1. youtube_search for the best tutorials (beginner to advanced); use youtube_transcript on the top 1-2 videos.
2. Write ONE document in this format:
   # <Topic>: Study Notes
   summary, key concepts, code examples
   # Learning Plan
   ## Week 1 / Week 2 / ... (4 weeks unless the user specifies)
   - [ ] daily tasks with a specific goal each
   - [ ] a YouTube video link per week
   - [ ] a mini project at the end of each week
   # Resources
   video links
3. Save it ONLY with {tool}. Never use any other save tool.
4. Reply with a 2-line summary and the link."""

def make_server(creds):
    return StdioServerParameters(command="python", args=["server.py"],
                                 env={**os.environ, **creds})

async def run_agent(history, dest, creds, gemini_key):
    llm = genai.Client(api_key=gemini_key)
    trace = []
    keep = DEST[dest]
    async with stdio_client(make_server(creds)) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            decls = [types.FunctionDeclaration(
                        name=t.name, description=t.description or "",
                        parameters_json_schema=t.inputSchema)
                     for t in (await session.list_tools()).tools
                     if t.name not in DEST.values() or t.name == keep]
            config = types.GenerateContentConfig(
                system_instruction=SYSTEM.format(tool=keep),
                tools=[types.Tool(function_declarations=decls)])
            while True:
                resp = llm.models.generate_content(
                    model=MODEL, contents=history, config=config)
                content = resp.candidates[0].content
                history.append(content)
                calls = [p.function_call for p in content.parts if p.function_call]
                if not calls:
                    text = "".join(p.text for p in content.parts if p.text)
                    return text, trace
                parts = []
                for c in calls:
                    out = await session.call_tool(c.name, dict(c.args))
                    txt = "".join(x.text for x in out.content if x.type == "text")
                    trace.append((c.name, dict(c.args), txt))
                    parts.append(types.Part.from_function_response(
                        name=c.name, response={"result": txt}))
                history.append(types.Content(role="user", parts=parts))

st.title("🎓 Learning Agent")

if "APP_PASSWORD" in os.environ:
    if st.sidebar.text_input("🔒 App password", type="password") != os.environ["APP_PASSWORD"]:
        st.info("Enter the app password to continue.")
        st.stop()

st.sidebar.header("🔑 Credentials")
gemini_key = st.sidebar.text_input("✨ Gemini API key", type="password")
yt_key = st.sidebar.text_input("▶️ YouTube API key", type="password")
dest = st.sidebar.radio("💾 Save output to", list(DEST.keys()))
if dest == "Notion":
    creds = {"NOTION_TOKEN": st.sidebar.text_input("📓 Notion integration token", type="password"),
             "NOTION_PARENT_PAGE_ID": st.sidebar.text_input("📄 Notion parent page ID")}
else:
    creds = {"DRIVE_ACCESS_TOKEN": st.sidebar.text_input("📁 Google access token", type="password")}
creds["YOUTUBE_API_KEY"] = yt_key

st.session_state.setdefault("api_msgs", [])
st.session_state.setdefault("chat", [])

for m in st.session_state.chat:
    avatar = "🧑" if m["role"] == "user" else "🤖"
    with st.chat_message(m["role"], avatar=avatar):
        st.markdown(m["text"])
        for name, args, out in m.get("trace", []):
            with st.expander(f"🔧 {name}"):
                st.json(args); st.code(out)

if q := st.chat_input("💬 Ask about any programming topic..."):
    if not gemini_key or not all(creds.values()):
        st.error("⚠️ Fill in all the sidebar fields first.")
        st.stop()
    st.session_state.chat.append({"role": "user", "text": q})
    with st.chat_message("user", avatar="🧑"):
        st.markdown(q)
    st.session_state.api_msgs.append(
        types.Content(role="user", parts=[types.Part(text=q)]))
    with st.chat_message("assistant", avatar="🤖"):
        with st.spinner("🤔 Thinking..."):
            try:
                text, trace = asyncio.run(
                    run_agent(st.session_state.api_msgs, dest, creds, gemini_key))
            except Exception as e:
                st.error(f"❌ Request failed: {e}")
                st.stop()
        st.markdown(text)
        for name, args, out in trace:
            with st.expander(f"🔧 {name}"):
                st.json(args); st.code(out)
    st.session_state.chat.append({"role": "assistant", "text": text, "trace": trace})

Writing app.py


In [ ]:
import subprocess, time, re
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true > st.log 2>&1", shell=True)
subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 > cf.log 2>&1", shell=True)
time.sleep(8)
print(re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read()).group(0))

https://hundred-yorkshire-artistic-moore.trycloudflare.com
